# Data Cleaning and Preparation

## Objective

Prepare the Olist datasets for exploratory analysis by applying documented business rules, handling missing values, selecting valid transactions, and creating an analysis-ready order-item-level dataset.

In [1]:
import pandas as pd

customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")

In [2]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(orders[column])

In [3]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


## 1. Filter Valid Transactions

For baseline revenue and business-performance analysis, only delivered orders are retained. Canceled, unavailable, and incomplete orders are excluded because they should not be treated as completed transactions.

The original raw datasets remain unchanged.

In [4]:
orders_clean = orders[
    orders["order_status"] == "delivered"
].copy()

In [5]:
print("Original orders:", orders.shape[0])
print("Delivered orders:", orders_clean.shape[0])
print("Removed orders:", orders.shape[0] - orders_clean.shape[0])

Original orders: 99441
Delivered orders: 96478
Removed orders: 2963


In [6]:
orders_clean["order_status"].value_counts()

order_status
delivered    96478
Name: count, dtype: int64

In [7]:
orders_clean["order_id"].duplicated().sum()

np.int64(0)

## 2. Handle Missing Product Categories

Products with missing category information are retained because they may still have valid transactions. Missing category values are labeled as `unknown` to allow consistent category-level analysis.

In [8]:
products_clean = products.copy()

products_clean["product_category_name"] = (
    products_clean["product_category_name"]
    .fillna("unknown")
)

In [9]:
print(
    "Missing categories before:",
    products["product_category_name"].isnull().sum()
)

print(
    "Missing categories after:",
    products_clean["product_category_name"].isnull().sum()
)

Missing categories before: 610
Missing categories after: 0


In [10]:
print("Original products:", products.shape[0])
print("Clean products:", products_clean.shape[0])

Original products: 32951
Clean products: 32951


## 3. Create Analysis-Ready Dataset

The cleaned datasets are joined to create a single order-item-level analytical dataset.

The final grain is:

**One row = one item position within a delivered order.**

In [11]:
analysis_df = (
    orders_clean
    .merge(
        customers,
        on="customer_id",
        how="left",
        validate="one_to_one"
    )
    .merge(
        order_items,
        on="order_id",
        how="inner",
        validate="one_to_many"
    )
    .merge(
        products_clean,
        on="product_id",
        how="left",
        validate="many_to_one"
    )
)

In [12]:
print("Rows:", analysis_df.shape[0])
print("Columns:", analysis_df.shape[1])

print(
    "Unique orders:",
    analysis_df["order_id"].nunique()
)

print(
    "Duplicate composite keys:",
    analysis_df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

Rows: 110197
Columns: 26
Unique orders: 96478
Duplicate composite keys: 0


In [13]:
analysis_df.isnull().sum().sort_values(ascending=False)

product_photos_qty               1537
product_description_lenght       1537
product_name_lenght              1537
product_weight_g                   18
product_width_cm                   18
product_length_cm                  18
product_height_cm                  18
order_approved_at                  15
order_delivered_customer_date       8
order_delivered_carrier_date        2
customer_zip_code_prefix            0
customer_unique_id                  0
order_estimated_delivery_date       0
order_status                        0
order_purchase_timestamp            0
customer_id                         0
order_id                            0
customer_city                       0
freight_value                       0
price                               0
shipping_limit_date                 0
seller_id                           0
product_id                          0
order_item_id                       0
customer_state                      0
product_category_name               0
dtype: int64

In [14]:
analysis_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,29.99,8.72,utilidades_domesticas,40.0,268.0,4.0,500.0,19.0,8.0,13.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,118.70,22.76,perfumaria,29.0,178.0,1.0,400.0,19.0,13.0,19.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,159.90,19.22,automotivo,46.0,232.0,1.0,420.0,24.0,19.0,21.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,45.00,27.20,pet_shop,59.0,468.0,3.0,450.0,30.0,10.0,20.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,19.90,8.72,papelaria,38.0,316.0,4.0,250.0,51.0,15.0,15.0


In [15]:
analysis_df.isnull().sum()[
    analysis_df.isnull().sum() > 0
].sort_values(ascending=False)

product_description_lenght       1537
product_name_lenght              1537
product_photos_qty               1537
product_length_cm                  18
product_height_cm                  18
product_weight_g                   18
product_width_cm                   18
order_approved_at                  15
order_delivered_customer_date       8
order_delivered_carrier_date        2
dtype: int64

## 4. Create Analysis Features

Additional columns are derived from the cleaned transaction data to support time-based, customer, product, regional, and freight analysis.

In [16]:
analysis_df["purchase_year"] = analysis_df["order_purchase_timestamp"].dt.year

analysis_df["purchase_month"] = analysis_df["order_purchase_timestamp"].dt.month

analysis_df["purchase_year_month"] = (
    analysis_df["order_purchase_timestamp"].dt.to_period("M").astype(str)
)

analysis_df["delivery_days"] = (
    analysis_df["order_delivered_customer_date"]
    - analysis_df["order_purchase_timestamp"]
).dt.days

analysis_df["freight_to_price_pct"] = (
    analysis_df["freight_value"]
    / analysis_df["price"]
    * 100
)

In [17]:
final_df = analysis_df[
    [
        "order_id",
        "order_item_id",
        "customer_unique_id",
        "order_purchase_timestamp",
        "purchase_year",
        "purchase_month",
        "purchase_year_month",
        "customer_city",
        "customer_state",
        "product_id",
        "product_category_name",
        "seller_id",
        "price",
        "freight_value",
        "freight_to_price_pct",
        "delivery_days"
    ]
].copy()

In [18]:
print("Final shape:", final_df.shape)

print(
    "Duplicate composite keys:",
    final_df.duplicated(
        subset=["order_id", "order_item_id"]
    ).sum()
)

print("Missing values:")
print(final_df.isnull().sum())

Final shape: (110197, 16)
Duplicate composite keys: 0
Missing values:
order_id                    0
order_item_id               0
customer_unique_id          0
order_purchase_timestamp    0
purchase_year               0
purchase_month              0
purchase_year_month         0
customer_city               0
customer_state              0
product_id                  0
product_category_name       0
seller_id                   0
price                       0
freight_value               0
freight_to_price_pct        0
delivery_days               8
dtype: int64


In [19]:
final_df.to_csv(
    "../data/processed/cleaned_order_items.csv",
    index=False
)

In [20]:
pd.read_csv(
    "../data/processed/cleaned_order_items.csv"
).shape

(110197, 16)